In [0]:
# MAGIC Databricks Gold Layer
# MAGIC
# MAGIC Ports Fabric's `nb_gold.ipynb` SCD2 dimensional model verbatim - same
# MAGIC `build_dim_policyholder`/`build_dim_vehicle`/`build_fact_claims`/`build_fact_telematics`
# MAGIC logic and the same SCD2 merge algorithm. Databricks runs on the same Delta Lake
# MAGIC engine as Fabric, so `DeltaTable.merge()` works identically - the only change is
# MAGIC addressing: Fabric's OneLake path-based tables ("Tables/dbo/dim_policyholder")
# MAGIC become Unity Catalog table names here (`DeltaTable.forName`/`saveAsTable`),
# MAGIC matching how bronze/silver already addressed everything on this platform.
# MAGIC
# MAGIC Run **after** `nb_generate_databricks` (needs `claim_events` and
# MAGIC `synthetic_telematics_fleet`). Job cluster, same as the rest of this build.

In [0]:
dbutils.widgets.text("catalog", "policybench_dev")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
from pyspark.sql import DataFrame, functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

In [0]:
def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
POLICYHOLDER_TRACKED_COLS = [
    "GENDER", "EDUCATION", "OCCUPATION", "INCOME", "HOME_VAL", "MSTATUS",
    "PARENT1", "HOMEKIDS", "KIDSDRIV", "URBANICITY", "YOJ",
]
 
VEHICLE_TRACKED_COLS = ["CAR_TYPE", "CAR_USE", "CAR_AGE", "BLUEBOOK", "RED_CAR", "TIF", "TRAVTIME"]

In [0]:
def _with_change_hash(df: DataFrame, tracked_cols: list) -> DataFrame:
    return df.withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in tracked_cols]), 256))

In [0]:
def build_dim_policyholder(silver_df: DataFrame) -> DataFrame:
    person_cols = ["ID", "BIRTH_DATE", "AGE"] + POLICYHOLDER_TRACKED_COLS
    dim = silver_df.select(*person_cols).dropDuplicates(["ID"])
    dim = _with_change_hash(dim, POLICYHOLDER_TRACKED_COLS)
    return dim

In [0]:
def build_dim_vehicle(silver_df: DataFrame) -> DataFrame:
    vehicle_cols = ["POLICY_ID", "ID"] + VEHICLE_TRACKED_COLS
    dim = silver_df.select(*vehicle_cols)
    dim = _with_change_hash(dim, VEHICLE_TRACKED_COLS)
    return dim

In [0]:
def scd2_merge(spark, table_name: str, new_df: DataFrame, business_key: str):
    """
    Standard SCD2 upsert, addressed by Unity Catalog table name (Fabric's version
    used a OneLake path - DeltaTable.forName/saveAsTable is the Databricks-native
    equivalent, same merge semantics underneath).
    """
    if not spark.catalog.tableExists(table_name):
        first_load = (
            new_df.withColumn("effective_start", F.current_timestamp())
            .withColumn("effective_end", F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .withColumn("version", F.lit(1))
        )
        first_load.write.format("delta").mode("overwrite").saveAsTable(table_name)
        return
 
    target = DeltaTable.forName(spark, table_name)
 
    (
        target.alias("t")
        .merge(new_df.alias("s"), f"t.{business_key} = s.{business_key} AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": F.lit(False), "effective_end": F.current_timestamp()},
        )
        .execute()
    )
 
    current = target.toDF().filter("is_current = true")
    changed_or_new = new_df.alias("s").join(
        current.alias("t"), on=business_key, how="left_anti"
    )
 
    prior_versions = target.toDF().groupBy(business_key).agg(F.max("version").alias("prior_version"))
    changed_or_new = changed_or_new.join(prior_versions, on=business_key, how="left")
    changed_or_new = changed_or_new.withColumn(
        "next_version", F.coalesce(F.col("prior_version"), F.lit(0)) + 1
    )
 
    to_insert = (
        changed_or_new.withColumn("effective_start", F.current_timestamp())
        .withColumn("effective_end", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("version", F.col("next_version"))
        .drop("prior_version", "next_version")
    )
    to_insert.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
def build_fact_claims(claim_events_df: DataFrame, dim_vehicle: DataFrame) -> DataFrame:
    fact = claim_events_df.join(
        dim_vehicle.select("POLICY_ID", "ID"), on=["POLICY_ID", "ID"], how="left"
    )
    return fact.select("POLICY_ID", "ID", "claim_id", "claim_date", "claim_amount", "claim_type")

In [0]:
def build_fact_telematics(fleet_df: DataFrame) -> DataFrame:
    return fleet_df.withColumn("event_date", F.to_date(F.from_unixtime(F.col("timestamp") / 1000)))

In [0]:
start_dt = datetime.now(timezone.utc)
 
silver_policyholders = spark.read.table("silver_car_insurance_claim")
claim_events = spark.read.table("claim_events")
synthetic_fleet = spark.read.table("synthetic_telematics_fleet")

In [0]:
dim_policyholder_df = build_dim_policyholder(silver_policyholders)
dim_vehicle_df = build_dim_vehicle(silver_policyholders)
 
scd2_merge(spark, "dim_policyholder", dim_policyholder_df, business_key="ID")
scd2_merge(spark, "dim_vehicle", dim_vehicle_df, business_key="POLICY_ID")

In [0]:
dim_vehicle_current = spark.read.table("dim_vehicle").filter("is_current = true")
 
fact_claims_df = build_fact_claims(claim_events, dim_vehicle_current)
fact_claims_df.write.format("delta").mode("overwrite").saveAsTable("fact_claims")
 
fact_telematics_df = build_fact_telematics(synthetic_fleet)
fact_telematics_df.write.format("delta").mode("overwrite").saveAsTable("fact_telematics")
 
end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "gold", start_dt, end_dt)

In [0]:
# MAGIC %md
# MAGIC ## Sanity check against the Fabric build

In [0]:
dim_policyholder_check = spark.read.table("dim_policyholder")
dim_vehicle_check = spark.read.table("dim_vehicle")
fact_claims_check = spark.read.table("fact_claims")
fact_telematics_check = spark.read.table("fact_telematics")
 
print("dim_policyholder rows:", dim_policyholder_check.count())
print("dim_vehicle rows:", dim_vehicle_check.count())
print("fact_claims rows:", fact_claims_check.count())
print("fact_telematics rows:", fact_telematics_check.count())
print()
print("dim_policyholder all is_current=true on first load (should equal row count):",
      dim_policyholder_check.filter("is_current = true").count())
print("fact_claims rows with no matched POLICY_ID (should be 0):",
      fact_claims_check.filter(F.col("POLICY_ID").isNull()).count())